<a href="https://colab.research.google.com/github/Ayush-Singh-36/fnn_Pytorch_Model/blob/main/fnn_pytorch_model_script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Development

## Device-Agnostic code

In [2]:
import torch
from torch import cuda
if cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")
print(device)

cuda


## Calling the dataset from kaggle, directly here

In [3]:
import os
from google.colab import userdata
import sys
def custom_exit(status):
    print(f"Kaggle API tried to exit with status {status}. Ignoring for Colab environment.")
    # Optionally, raise an exception or log, instead of actual exit.
sys.exit = custom_exit
exit = custom_exit # Patch the global 'exit' function as well
try:
    __builtins__.exit = custom_exit # Explicitly patch built-in exit
except AttributeError:
    print("Could not patch __builtins__.exit - it might not be present or modifiable in this environment.")

# Retrieve credentials from Colab secrets
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# Now import and run the download code
from kaggle.api.kaggle_api_extended import KaggleApi

dataset_slug = "nudratabbas/software-developer-salary-prediction-dataset"
download_path = "./data"

print("Authenticating via environment variables...")
api = KaggleApi()
api.authenticate()

print("Downloading movie dataset from Kaggle...")
api.dataset_download_files(dataset_slug, path=download_path, unzip=True)

print(f"Done! Your files have been saved to the '{download_path}' folder.")


Authenticating via environment variables...
Dataset URL: https://www.kaggle.com/datasets/nudratabbas/software-developer-salary-prediction-dataset
Done! Your files have been saved to the './data' folder.


## Checking for cardinality

### data_dictionary.csv

In [4]:
import pandas as pd
import numpy as np
def profile_dataset_features(df: pd.DataFrame, max_categories_to_print: int = 10):
    """
    Scans a massive dataset to automatically break down columns into
    categorical option spaces or numerical statistical boundaries.
    """
    print(f"=== Dataset Shape: {df.shape[0]} rows | {df.shape[1]} columns ===\n")

    # 1. Separate column types automatically
    categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns
    numerical_cols = df.select_dtypes(include=[np.number]).columns

    print(f"Found {len(categorical_cols)} Categorical columns and {len(numerical_cols)} Numerical columns.\n")
    print("-" * 50)
    print("CATEGORICAL FEATURE OPTIONS DISCOVERY")
    print("-" * 50)

    # 2. Extract Options from Categorical Columns
    for col in categorical_cols:
        unique_vals = df[col].dropna().unique()
        num_unique = len(unique_vals)
        missing_count = df[col].isna().sum()

        print(f"\n🔹 Feature: '{col}' | Unique Values Count: {num_unique} | Missing: {missing_count} rows")

        # High cardinality warning (e.g., IDs, Hash keys, open text fields)
        if num_unique > 30:
            print(f"  ⚠️ High Cardinality! Showing first 5 options sample: {list(unique_vals[:5])}...")
        else:
            # Print value distributions so you know if an option is incredibly rare
            value_counts = df[col].value_counts(dropna=False)
            for val, count in value_counts.items():
                pct = (count / len(df)) * 100
                print(f"  - [{val}]: {count} occurrences ({pct:.2f}%)")

    print("\n" + "-" * 50)
    print("NUMERICAL FEATURE BOUNDARY DISCOVERY")
    print("-" * 50)

    # 3. Extract Ranges from Numerical Columns
    # Using describe gives you min, max, and percentiles to spot extreme values or outliers
    if len(numerical_cols) > 0:
        numeric_summary = df[numerical_cols].describe().T[['min', 'max', 'mean']]
        display(numeric_summary)
    else:
        print("No numerical columns to describe.")

# --- Example of running it on your data ---
df = pd.read_csv("./data/data_dictionary.csv")
profile_dataset_features(df)

=== Dataset Shape: 7 rows | 3 columns ===

Found 3 Categorical columns and 0 Numerical columns.

--------------------------------------------------
CATEGORICAL FEATURE OPTIONS DISCOVERY
--------------------------------------------------

🔹 Feature: 'Column' | Unique Values Count: 7 | Missing: 0 rows
  - [experience]: 1 occurrences (14.29%)
  - [country]: 1 occurrences (14.29%)
  - [education]: 1 occurrences (14.29%)
  - [languages]: 1 occurrences (14.29%)
  - [frameworks]: 1 occurrences (14.29%)
  - [company_size]: 1 occurrences (14.29%)
  - [salary_usd]: 1 occurrences (14.29%)

🔹 Feature: 'Type' | Unique Values Count: 3 | Missing: 0 rows
  - [string]: 5 occurrences (71.43%)
  - [number]: 1 occurrences (14.29%)
  - [target]: 1 occurrences (14.29%)

🔹 Feature: 'Description' | Unique Values Count: 7 | Missing: 0 rows
  - [Years of professional coding experience]: 1 occurrences (14.29%)
  - [Country of residence]: 1 occurrences (14.29%)
  - [Highest level of formal education]: 1 occurrenc

### train.csv

In [5]:
import pandas as pd
import numpy as np
def profile_dataset_features(df: pd.DataFrame, max_categories_to_print: int = 10):
    """
    Scans a massive dataset to automatically break down columns into
    categorical option spaces or numerical statistical boundaries.
    """
    print(f"=== Dataset Shape: {df.shape[0]} rows | {df.shape[1]} columns ===\n")

    # 1. Separate column types automatically
    categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns
    numerical_cols = df.select_dtypes(include=[np.number]).columns

    print(f"Found {len(categorical_cols)} Categorical columns and {len(numerical_cols)} Numerical columns.\n")
    print("-" * 50)
    print("CATEGORICAL FEATURE OPTIONS DISCOVERY")
    print("-" * 50)

    # 2. Extract Options from Categorical Columns
    for col in categorical_cols:
        unique_vals = df[col].dropna().unique()
        num_unique = len(unique_vals)
        missing_count = df[col].isna().sum()

        print(f"\n🔹 Feature: '{col}' | Unique Values Count: {num_unique} | Missing: {missing_count} rows")

        # High cardinality warning (e.g., IDs, Hash keys, open text fields)
        if num_unique > 30:
            print(f"  ⚠️ High Cardinality! Showing first 5 options sample: {list(unique_vals[:5])}...")
        else:
            # Print value distributions so you know if an option is incredibly rare
            value_counts = df[col].value_counts(dropna=False)
            for val, count in value_counts.items():
                pct = (count / len(df)) * 100
                print(f"  - [{val}]: {count} occurrences ({pct:.2f}%)")

    print("\n" + "-" * 50)
    print("NUMERICAL FEATURE BOUNDARY DISCOVERY")
    print("-" * 50)

    # 3. Extract Ranges from Numerical Columns
    # Using describe gives you min, max, and percentiles to spot extreme values or outliers
    if len(numerical_cols) > 0:
        numeric_summary = df[numerical_cols].describe().T[['min', 'max', 'mean']]
        display(numeric_summary)
    else:
        print("No numerical columns to describe.")

# --- Example of running it on your data ---
train_df = pd.read_csv("./data/train.csv")
profile_dataset_features(train_df)

=== Dataset Shape: 40000 rows | 7 columns ===

Found 5 Categorical columns and 2 Numerical columns.

--------------------------------------------------
CATEGORICAL FEATURE OPTIONS DISCOVERY
--------------------------------------------------

🔹 Feature: 'country' | Unique Values Count: 10 | Missing: 0 rows
  - [USA]: 16003 occurrences (40.01%)
  - [UK]: 4013 occurrences (10.03%)
  - [Canada]: 4003 occurrences (10.01%)
  - [Germany]: 3988 occurrences (9.97%)
  - [India]: 3970 occurrences (9.93%)
  - [Australia]: 2064 occurrences (5.16%)
  - [France]: 2031 occurrences (5.08%)
  - [Japan]: 1918 occurrences (4.79%)
  - [Brazil]: 1030 occurrences (2.57%)
  - [Singapore]: 980 occurrences (2.45%)

🔹 Feature: 'education' | Unique Values Count: 5 | Missing: 0 rows
  - [Bachelors]: 20061 occurrences (50.15%)
  - [Masters]: 11968 occurrences (29.92%)
  - [Some College]: 4005 occurrences (10.01%)
  - [High School]: 1988 occurrences (4.97%)
  - [PhD]: 1978 occurrences (4.95%)

🔹 Feature: 'languages'

,min,max,mean
experience,0.0,40.0,19.912875
salary_usd,12024.0,277554.0,131834.441525


### test.csv

In [6]:
import pandas as pd
import numpy as np
def profile_dataset_features(df: pd.DataFrame, max_categories_to_print: int = 10):
    """
    Scans a massive dataset to automatically break down columns into
    categorical option spaces or numerical statistical boundaries.
    """
    print(f"=== Dataset Shape: {df.shape[0]} rows | {df.shape[1]} columns ===\n")

    # 1. Separate column types automatically
    categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns
    numerical_cols = df.select_dtypes(include=[np.number]).columns

    print(f"Found {len(categorical_cols)} Categorical columns and {len(numerical_cols)} Numerical columns.\n")
    print("-" * 50)
    print("CATEGORICAL FEATURE OPTIONS DISCOVERY")
    print("-" * 50)

    # 2. Extract Options from Categorical Columns
    for col in categorical_cols:
        unique_vals = df[col].dropna().unique()
        num_unique = len(unique_vals)
        missing_count = df[col].isna().sum()

        print(f"\n🔹 Feature: '{col}' | Unique Values Count: {num_unique} | Missing: {missing_count} rows")

        # High cardinality warning (e.g., IDs, Hash keys, open text fields)
        if num_unique > 30:
            print(f"  ⚠️ High Cardinality! Showing first 5 options sample: {list(unique_vals[:5])}...")
        else:
            # Print value distributions so you know if an option is incredibly rare
            value_counts = df[col].value_counts(dropna=False)
            for val, count in value_counts.items():
                pct = (count / len(df)) * 100
                print(f"  - [{val}]: {count} occurrences ({pct:.2f}%)")

    print("\n" + "-" * 50)
    print("NUMERICAL FEATURE BOUNDARY DISCOVERY")
    print("-" * 50)

    # 3. Extract Ranges from Numerical Columns
    # Using describe gives you min, max, and percentiles to spot extreme values or outliers
    if len(numerical_cols) > 0:
        numeric_summary = df[numerical_cols].describe().T[['min', 'max', 'mean']]
        display(numeric_summary)
    else:
        print("No numerical columns to describe.")

# --- Example of running it on your data ---
test_df = pd.read_csv("./data/test.csv")
profile_dataset_features(test_df)

=== Dataset Shape: 10000 rows | 7 columns ===

Found 5 Categorical columns and 2 Numerical columns.

--------------------------------------------------
CATEGORICAL FEATURE OPTIONS DISCOVERY
--------------------------------------------------

🔹 Feature: 'country' | Unique Values Count: 10 | Missing: 0 rows
  - [USA]: 3999 occurrences (39.99%)
  - [Germany]: 1044 occurrences (10.44%)
  - [India]: 1037 occurrences (10.37%)
  - [UK]: 997 occurrences (9.97%)
  - [Canada]: 935 occurrences (9.35%)
  - [Japan]: 516 occurrences (5.16%)
  - [Australia]: 506 occurrences (5.06%)
  - [France]: 502 occurrences (5.02%)
  - [Brazil]: 251 occurrences (2.51%)
  - [Singapore]: 213 occurrences (2.13%)

🔹 Feature: 'education' | Unique Values Count: 5 | Missing: 0 rows
  - [Bachelors]: 5077 occurrences (50.77%)
  - [Masters]: 2995 occurrences (29.95%)
  - [Some College]: 990 occurrences (9.90%)
  - [High School]: 481 occurrences (4.81%)
  - [PhD]: 457 occurrences (4.57%)

🔹 Feature: 'languages' | Unique Val

,min,max,mean
experience,0.0,40.0,19.7701
salary_usd,12032.0,280900.0,131110.6842


## Creating dataloader out of dataframes

In [7]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

class MyDataset(Dataset):
  def __init__(self, dataframe):

    labels_df = dataframe["salary_usd"].astype(float)
    features_df = dataframe.drop(columns = ["salary_usd"])

    categorical_cols = features_df.select_dtypes(include=['object', 'category']).columns

    if not categorical_cols.empty:
      features_df = pd.get_dummies(features_df, columns=categorical_cols, drop_first=True)

    features_df = features_df.apply(pd.to_numeric, errors='coerce').fillna(0)

    self.features = torch.tensor(features_df.to_numpy(dtype=np.float32))
    self.labels = torch.tensor(labels_df.to_numpy(dtype=np.float32))

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, idx):
    return self.features[idx], self.labels[idx]

train_dataset = MyDataset(train_df)
test_dataset = MyDataset(test_df)
train_dataloader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
test_dataloader = DataLoader(test_dataset, batch_size = 32, shuffle = False)

**Calculating number of dimensions to embbed**

In [10]:
import pandas as pd
import numpy as np
categorical_cols = train_df.select_dtypes(include = ["object", "category"]).columns
emb_dim_num = [
    (train_df[col].nunique(), min(50, train_df[col].nunique() // 2))
    for col in categorical_cols
]
print(emb_dim_num)

[(10, 5), (5, 2), (100, 50), (100, 50), (6, 3)]


**Encoding**

In [12]:
import torch
import torch.nn as nn
embedding_layer = nn.ModuleList([
    nn.Embedding(num_embeddings = 7, embedding_dim = emb_dim)
    for num_embeddings, emb_dim in emb_dim_num
])